# Deploying NVIDIA Nemotron-3-Super with TensorRT LLM

This notebook will walk you through how to run the `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B` model via TensorRT-LLM.

[TensorRT LLM](https://nvidia.github.io/TensorRT-LLM/) is NVIDIA’s open-source library for accelerating and optimizing LLM inference performance on NVIDIA GPUs. 

For more details on the model [click here](https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16).

Prerequisites:
- NVIDIA GPU with recent drivers (>= 264 GB VRAM for BF16, >= 160 GB VRAM for FP8) and CUDA 12.x.
- Python 3.10+
- TensorRT-LLM (you can refer to NVIDIA documentation, or pull this [container](https://catalog.ngc.nvidia.com/orgs/nvidia/teams/tensorrt-llm/containers/release?version=1.3.0rc4))


#### Launch on NVIDIA Brev
You can simplify the environment setup by using [NVIDIA Brev](https://developer.nvidia.com/brev). Click the button to launch this project on a Brev instance with the necessary dependencies pre-configured.

Once deployed, click on the "Open Notebook" button to get started with this guide. 

**For BF16 (4x H100):**

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-39uG7hJrEpjbyDDPzCruj2SRVEi)

**For FP8 (2x H100):**

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-3AjvN05TVaeGawyDnxSX6Sov4UV)

## Prerequisites & environment

Set up a containerized environment for TensorRT-LLM by running the following in a **host** terminal.

For example, on Brev you can set `CACHE_ROOT=/ephemeral` for model cache and temp files so the container does not run out of space.
If you are not on Brev, set `CACHE_ROOT` to a writable path on your machine.

```shell
export CACHE_ROOT=/ephemeral
mkdir -p "$CACHE_ROOT/trtllm_cache"
mkdir -p "$CACHE_ROOT/trtllm_tmp"


docker run --rm -it --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 --gpus=all \
  -p 8000:8000 \
  -v "$CACHE_ROOT":"$CACHE_ROOT" \
  -e HF_HOME="$CACHE_ROOT/trtllm_cache" \
  -e HUGGINGFACE_HUB_CACHE="$CACHE_ROOT/trtllm_cache" \
  -e TMPDIR="$CACHE_ROOT/trtllm_tmp" \
  -e TEMP="$CACHE_ROOT/trtllm_tmp" \
  -e TMP="$CACHE_ROOT/trtllm_tmp" \
  nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc4
```

You now have TRT-LLM set up!

In [ ]:
#If pip not found
!python -m ensurepip --default-pip

In [ ]:
%pip install torch==2.9.1 openai==2.6.1 requests

## Verify GPU

Check that CUDA is available and the GPU is detected correctly.


In [4]:
# Environment check
import sys
import torch

print(f"Python: {sys.version}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")


Python: 3.12.12 (main, Feb 12 2026, 00:42:14) [Clang 21.1.4 ]
CUDA available: True
Num GPUs: 4
GPU[0]: NVIDIA H100 PCIe
GPU[1]: NVIDIA H100 PCIe
GPU[2]: NVIDIA H100 PCIe
GPU[3]: NVIDIA H100 PCIe


## OpenAI-compatible server

Start a local OpenAI-compatible server with TensorRT-LLM via the terminal, within the running docker container.

Ensure that the following commands are executed from the docker terminal.

Choose the variant you want to serve:
- BF16: `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16` on 4x H100 (`--tp_size 4 --ep_size 4`)
- FP8: `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8` on 2x H100 (`--tp_size 2 --ep_size 2`)

### Create a YAML file with the required configuration

```shell
cat > ./extra-llm-api-config.yml << EOF
kv_cache_config:
  enable_block_reuse: false
  mamba_ssm_cache_dtype: float16
moe_config:
   backend: TRTLLM
cuda_graph_config:
    enable_padding: true
    batch_sizes: [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
EOF
```

### Load the model

#### BF16 (4x H100)

```shell
mpirun -n 1 --allow-run-as-root --oversubscribe \
trtllm-serve nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16 \
  --host 0.0.0.0 \
  --port 8000 \
  --backend pytorch \
  --max_batch_size 128 \
  --tp_size 4 --ep_size 4 \
  --max_num_tokens 16384 \
  --trust_remote_code \
  --reasoning_parser nano-v3 \
  --tool_parser qwen3_coder \
  --extra_llm_api_options extra-llm-api-config.yml
```

#### FP8 (2x H100)

```shell
mpirun -n 1 --allow-run-as-root --oversubscribe \
trtllm-serve nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8 \
  --host 0.0.0.0 \
  --port 8000 \
  --backend pytorch \
  --max_batch_size 128 \
  --tp_size 2 --ep_size 2 \
  --max_num_tokens 16384 \
  --trust_remote_code \
  --reasoning_parser nano-v3 \
  --tool_parser qwen3_coder \
  --extra_llm_api_options extra-llm-api-config.yml
```


Your server is now running!

### Use the API

Use the OpenAI-compatible client to send requests to the TensorRT-LLM server.

Note: The model supports two modes - Reasoning ON (default) vs OFF. This can be toggled by setting enable_thinking to False, as shown below. 

In [ ]:
from openai import OpenAI
import requests

# Setup client
BASE_URL = "http://0.0.0.0:8000/v1"
API_KEY = "null" 
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

model_id = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16"  # BF16
# model_id = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8"  # FP8

In [12]:
# Reasoning on (default)
print("Reasoning on")
response = client.chat.completions.create(
    model=model_id,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1,
    max_tokens=1024,
)
print(response.choices[0].message.reasoning_content, response.choices[0].message.content)

print("\n")

# Reasoning off
print("Reasoning off")
response = client.chat.completions.create(
    model=model_id,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 bullet points about TensorRT-LLM."}
    ],
    temperature=0,
    max_tokens=256,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)
print(response.choices[0].message.reasoning_content, response.choices[0].message.content)

Reasoning on
Okay, user wants a haiku about GPUs. Hmm, they're probably a tech enthusiast or developer who appreciates both poetry and hardware. The challenge is capturing GPU essence in 5-7-5 syllables while making it vivid. 

First, gotta nail the core GPU functions: parallel processing, rendering graphics, massive computation. But haiku can't be technical - needs natural imagery. "Silent" feels right for idle GPUs, "boom" for when they kick in. 

Wait, should I mention "cuda" or "shaders"? No, too jargon-heavy for poetry. "Cores" is perfect - simple but precise. "Cycles" implies the rhythm of processing. 

*checks syllable count* 
"Silent cores awaken" (5) 
"Lightning cycles paint the screen" (7 - "lightning" as 2 syllables, "paint" as 1) 
"Endless data flows" (5) 

*double-checks* 
Yep, "endless" is 2, "data" is 2, "flows" is 1. Nailed it. User'll probably smile at "lightning" - it's techy but poetic. Hope they like the "sinuous" touch too; shows how data moves like liquid light. 


In [13]:
# Streaming chat completion
print("Streaming response:")
stream = client.chat.completions.create(
    model=model_id,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"}
    ],
    temperature=0.7,
    max_tokens=1024,
    stream=True,
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

Streaming response:

The first five prime numbers are:

1. 2  
2. 3  
3. 5  
4. 7  
5. 11

### Tool calling

Use the OpenAI tools schema to call functions via the TensorRT-LLM endpoint.

In [14]:
# Tool calling via OpenAI tools schema
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate_tip",
            "parameters": {
                "type": "object",
                "properties": {
                    "bill_total": {
                        "type": "integer",
                        "description": "The total amount of the bill"
                    },
                    "tip_percentage": {
                        "type": "integer",
                        "description": "The percentage of tip to be applied"
                    }
                },
                "required": ["bill_total", "tip_percentage"]
            }
        }
    }
]

completion = client.chat.completions.create(
    model=model_id,
    messages=[
        {"role": "system", "content": ""},
        {"role": "user", "content": "My bill is $50. What will be the amount for 15% tip?"}
    ],
    tools=TOOLS,
    temperature=0.6,
    top_p=0.95,
    max_tokens=512,
    stream=False
)

print(completion.choices[0].message.reasoning_content, completion.choices[0].message.content)
print(completion.choices[0].message.tool_calls)

The user wants to calculate a 15% tip on a $50 bill. I need to use the calculate_tip function. The function requires bill_total and tip_percentage parameters. Bill total is 50, tip percentage is 15. I'll call the function.
 


[ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-d2783abb5b494d77b07c9212dd0f011c', function=Function(arguments='{"bill_total": 50, "tip_percentage": 15}', name='calculate_tip'), type='function')]


### Controlling Reasoning Budget

The `reasoning_budget` parameter allows you to limit the length of the model's reasoning trace. When the reasoning output reaches the specified token budget, the model will attempt to gracefully end the reasoning at the next newline character. 

If no newline is encountered within 500 tokens after reaching the budget threshold, the reasoning trace will be forcibly terminated at `reasoning_budget + 500` tokens to prevent excessive generation.


In [ ]:
from typing import Any, Dict, List
import openai
from transformers import AutoTokenizer


class ThinkingBudgetClient:
    def __init__(self, base_url: str, api_key: str, tokenizer_name_or_path: str):
        self.base_url = base_url
        self.api_key = api_key
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path)
        self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key)

    def chat_completion(
        self,
        model: str,
        messages: List[Dict[str, Any]],
        reasoning_budget: int = 512,
        max_tokens: int = 1024,
        **kwargs,
    ) -> Dict[str, Any]:
        assert (
            max_tokens > reasoning_budget
        ), f"reasoning_budget must be smaller than max_tokens. Given {max_tokens=} and {reasoning_budget=}"

        # 1. first call chat completion to get reasoning content
        response = self.client.chat.completions.create(
            model=model, 
            messages=messages, 
            max_tokens=reasoning_budget, 
            **kwargs
        )
        
        reasoning_content = response.choices[0].message.reasoning_content or ""
        
        if "</think>" not in reasoning_content:
            # reasoning content is too long, closed with a period (.)
            reasoning_content = f"{reasoning_content}.\n</think>\n\n"
        
        reasoning_tokens_used = len(
            self.tokenizer.encode(reasoning_content, add_special_tokens=False)
        )
        remaining_tokens = max_tokens - reasoning_tokens_used
        
        assert (
            remaining_tokens > 0
        ), f"remaining tokens must be positive. Given {remaining_tokens=}. Increase max_tokens or lower reasoning_budget."

        # 2. append reasoning content to messages and call completion
        messages.append({"role": "assistant", "content": reasoning_content})
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            continue_final_message=True,
        )
        
        response = self.client.completions.create(
            model=model, 
            prompt=prompt, 
            max_tokens=remaining_tokens, 
            **kwargs
        )

        response_data = {
            "reasoning_content": reasoning_content.strip().strip("</think>").strip(),
            "content": response.choices[0].text,
            "finish_reason": response.choices[0].finish_reason,
        }
        return response_data

In [ ]:
# Client
model_id = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16"  # BF16
# model_id = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8"  # FP8

client = ThinkingBudgetClient(
    base_url="http://0.0.0.0:8000/v1",
    api_key="null",
    tokenizer_name_or_path=model_id
)

In [12]:
resp = client.chat_completion(
    model=model_id,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1,
    max_tokens=512,
    reasoning_budget=128
)
print("Reasoning:", resp["reasoning_content"], "\nContent:", resp["content"])

Reasoning: We need to write a haiku about GPUs. A haiku is 5-7-5 syllable structure. Provide 3 lines, 5 syllables, 7 syllables, 5 syllables. Might mention graphics, processing, cores, etc. Provide a nice poetic haiku. Ensure correct syllable count.

Let's craft: "Silicon veins pulse / Parallel rivers compute dreams / Light builds worlds anew"

Check syllables:

Line1: "Silicon veins pulse" -> Sil-i-con (3) veins (1) pulse (1) = 5? Actually "Silicon" is 3 syllables (sil-i-con. 
Content: 
Silicon veins pulse  
Parallel rivers compute dreams  
Light builds worlds anew


## Cleanup and shutdown

To tear down this TensorRT-LLM workflow:

1. In the terminal running `trtllm-serve`, press `Ctrl+C` to stop the server.
2. In the Docker shell, run `exit` to stop the container (`--rm` removes it automatically).
3. Optionally run the next cell to clear notebook-side CUDA cache before your next run.

In [ ]:
import gc
import torch

# Optional notebook-side cleanup (server/container are stopped from terminal)
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Notebook-side CUDA cache cleanup complete.")